In [ ]:
pip install sentence-transformers pandas numpy

In [2]:
import os
import csv
import base64
import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()
from openai import OpenAI

# OpenAI client
client = OpenAI()

# 경로 설정
INPUT_PATH = "/Users/gieun/Desktop/PICKLE/3rd_project/database/sql/source_csv/review.csv"
OUTPUT_PATH = "/Users/gieun/Desktop/PICKLE/3rd_project/database/sql/db_csv/review.csv"

In [3]:
# 임베딩 생성 함수
def get_embedding(text: str):
    if text is None or text.strip() == "":
        text = " "  # 빈 값 방지

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

In [4]:

# embedding → base64 변환
def embedding_to_base64(embedding_vector):
    vec = np.array(embedding_vector, dtype=np.float32)
    blob = vec.tobytes()
    encoded = base64.b64encode(blob).decode("utf-8")
    return encoded

In [11]:
# CSV 처리
def process_csv():
    with open(INPUT_PATH, "r", encoding="utf-8") as infile:
        reader = csv.DictReader(infile)
        rows = list(reader)

    output_rows = []

    for row in tqdm(rows):
        # 어떤 컬럼을 embedding할지 선택
        # ERD 기준: review.content + menu
        menu = row.get("menu")
        content = row.get("content")

        text = ""
        if menu and content:
            text = f"{menu} {content}" 
        elif menu:
            text = menu
        elif content:
            text = content

        # embedding 생성
        embedding = get_embedding(text)

        # base64 변환
        encoded_embedding = embedding_to_base64(embedding)

        # 컬럼 추가
        row["embedding"] = encoded_embedding

        output_rows.append(row)

    # 저장
    fieldnames = list(output_rows[0].keys())

    with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as outfile:
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(output_rows)

    print("✅ 완료:", OUTPUT_PATH)


if __name__ == "__main__":
    process_csv()

100%|██████████| 422/422 [02:04<00:00,  3.39it/s]

✅ 완료: /Users/gieun/Desktop/PICKLE/3rd_project/database/sql/db_csv/review.csv


In [12]:
import csv

CSV_PATH = "/Users/gieun/Desktop/PICKLE/3rd_project/database/sql/db_csv/review.csv"

with open(CSV_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    row = next(reader)

encoded = row["embedding"]

blob = base64.b64decode(encoded)
vec = np.frombuffer(blob, dtype=np.float32)

print(len(vec))
print(vec[:5])

1536
[-0.02577209 -0.03918457 -0.05041504 -0.00599289  0.02680969]


In [14]:
CSV_PATH = "/Users/gieun/Desktop/PICKLE/3rd_project/database/sql/db_csv/review.csv"


# ✅ embedding 생성
def get_embedding(text: str):
    if text is None or text.strip() == "":
        text = " "
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return np.array(response.data[0].embedding, dtype=np.float32)


# ✅ base64 → numpy vector 복원
def decode_embedding(encoded_str):
    blob = base64.b64decode(encoded_str)
    vec = np.frombuffer(blob, dtype=np.float32)
    return vec


# ✅ cosine similarity
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# ✅ 검색 함수
def search_similar_reviews(query, top_k=5):
    query_vec = get_embedding(query)

    results = []

    with open(CSV_PATH, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            if "embedding" not in row or not row["embedding"]:
                continue

            review_vec = decode_embedding(row["embedding"])

            score = cosine_similarity(query_vec, review_vec)

            results.append((score, row))

    # 유사도 높은 순 정렬
    results.sort(key=lambda x: x[0], reverse=True)

    return results[:top_k]


# ✅ 실행 예시
if __name__ == "__main__":
    query = "매콤한 떡볶이"
    results = search_similar_reviews(query, top_k=5)

    for score, row in results:
        print(f"[유사도: {score:.4f}] {row.get('content')}")

[유사도: 0.5543] 초등학생 시절에 맛있게 먹은 떡볶이가 생각나는 맛
사장님도 친절해요
[유사도: 0.5345] 1인세트(?)처럼 다양하게 먹을 수 있는 메뉴가 있어서ㅜ좋으나
튀김은 너무 바싹 튀겨져 튀김옷이 딱딱해요. 어묵은 덜 불렸고요. 순대도 그냥...떡볶이도 그냥..
[유사도: 0.5202] 신대방삼거리 성대시장에 있는 떡볶이 집입니다
맛있고 가성비 있어요
[유사도: 0.5177] 어렸을 때 자주먹던 시장 떡볶이 맛이예요. 가격도 비싸지 않고 맛있어요
[유사도: 0.5122] 떡튀순세트 포장해서 먹었어요! 딱 상상했던 시장 떡볶이 맛!


In [18]:
CSV_PATH = "/Users/gieun/Desktop/PICKLE/3rd_project/database/sql/db_csv/review.csv"


# ✅ embedding 생성
def get_embedding(text: str):
    if text is None or text.strip() == "":
        text = " "
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return np.array(response.data[0].embedding, dtype=np.float32)


# ✅ base64 → numpy vector 복원
def decode_embedding(encoded_str):
    blob = base64.b64decode(encoded_str)
    vec = np.frombuffer(blob, dtype=np.float32)
    return vec


# ✅ cosine similarity
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# ✅ 검색 함수
def search_similar_reviews(query, top_k=5):
    query_vec = get_embedding(query)

    results = []

    with open(CSV_PATH, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            if "embedding" not in row or not row["embedding"]:
                continue

            review_vec = decode_embedding(row["embedding"])

            score = cosine_similarity(query_vec, review_vec)
    
            if score > 0.4:  # 0.4 이상인 것만 결과에 포함
                results.append((score, row))

    # 유사도 높은 순 정렬
    results.sort(key=lambda x: x[0], reverse=True)

    return results[:top_k]


# ✅ 실행 예시
if __name__ == "__main__":
    query = "가성비 좋은 맛집"
    results = search_similar_reviews(query, top_k=5)

    for score, row in results:
        print(f"[유사도: {score:.4f}] {row.get('content')}")

[유사도: 0.5640] 그닥 맛집은 아닌데 손님이 많은집
위치가 좋아서 그런가바요
[유사도: 0.5228] 가성비 좋은 술집입니다
아주 맛나고 사장님 음식솜씨 좋으셔서 맛나게 잘먹고 왔어요
[유사도: 0.5093] 신대방삼거리 성대시장에 있는 떡볶이 집입니다
맛있고 가성비 있어요
[유사도: 0.4999] 신대방삼거리 깔끔한 돈까스 일식 맛집~ 맥주랑 같이 먹으면 아주 맛이뜸🤍
[유사도: 0.4991] 가게는 작지만 가성비대비 양도 많고 맛도
괜찮아요 또 올만한 곳이에요
